# Лабораторная работа 5

**Тема:** моделирование непрерывных случайных величин с заданными законами распределений.

**Выполнил:** Ипатенков Григорий Александрович, ИВТ-4, вариант 6

**Задание:**
1. Сформировать выборку равномерно распределённых на $(a, b)$ случайных величин (65) методом обратных функций.
2. Сформировать выборку величин, распределённых по гауссовскому закону $N(m, D^2)$, на основе ЦПТ.
3. Методом Неймана сформировать выборку величин, распределённых по усечённому рэлеевскому закону с параметром $b$ (69).
4. Объём выборки $N \ge 10^3$, параметры моделирования задаются при входе в программу.
5. Для каждой выборки построить гистограмму и полигон накопленных частот, нанести теоретические кривые.
6. Рассчитать выборочные средние и дисперсии по формулам (75)–(77) и сравнить с теоретическими значениями.

## Импорт библиотек

Базовые случайные числа $r_i \in (0, 1)$ берутся из генератора `numpy`. Как и описано в теории,
при `SEED = None` генератор инициализируется от системного таймера, поэтому при каждом запуске
получается новая псевдослучайная последовательность.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["figure.dpi"] = 100

## Параметры моделирования (вариант 6)

Параметры привязаны к номеру варианта $V = 6$:

| Параметр | Значение |
|---|---|
| Объём выборки $N$ | $1000 \cdot V = 6000$ |
| Равномерный закон, интервал $(a, b)$ | $(1,\ 1 + V) = (1,\ 7)$ |
| Гауссовский закон $N(m, D^2)$ | $m = V = 6,\ D = 2$ |
| Число слагаемых в ЦПТ $n$ | 12 |
| Рэлеевский закон, параметр $b$ | 1.5 |
| Правая граница усечения $x_{max}$ | $3b = 4.5$ |

По заданию параметры задаются при входе в программу: при запуске она спрашивает каждый параметр.
Пустой ввод (Enter) оставляет значение варианта 6 из таблицы. `ASK_INPUT = False` отключает вопросы.

In [ ]:
VARIANT = 6
ASK_INPUT = True    # False — без вопросов, сразу параметры варианта 6
SEED = None         # None — от таймера; число — воспроизводимый результат


def ask(prompt, default, cast=float):
    if not ASK_INPUT:
        return default
    s = input(f"{prompt} [{default}]: ").strip()
    return cast(s) if s else default


N = ask("Объём выборки N (>= 1000)", 1000 * VARIANT, int)

# Равномерное распределение на (a, b)
a = ask("Равномерный закон: a", 1.0)
b = ask("Равномерный закон: b", 1.0 + VARIANT)

# Гауссовское распределение N(m, D^2)
m = ask("Гауссовский закон: m", float(VARIANT))
D = ask("Гауссовский закон: D (СКО)", 2.0)
n_clt = ask("ЦПТ: число слагаемых n (6 или 12)", 12, int)

# Усечённое рэлеевское распределение на (0, x_max)
b_ray = ask("Рэлеевский закон: b", 1.5)
x_max = ask("Рэлеевский закон: граница усечения x_max", 3 * b_ray)

assert N >= 1000, "по заданию N >= 10^3"
assert b > a and D > 0 and b_ray > 0 and x_max > 0

rng = np.random.default_rng(SEED)

print(f"N = {N}")
print(f"Равномерный:  (a, b) = ({a}, {b})")
print(f"Гауссовский:  m = {m}, D = {D}, n = {n_clt}")
print(f"Рэлеевский:   b = {b_ray}, x_max = {x_max}")

## Выборочные моменты

* Первый выборочный момент (выборочное среднее), формула (75):
  $m_x = \frac{1}{N}\sum_{i=1}^{N} x_i$
* Выборочная дисперсия при **известном** математическом ожидании $m$, формула (76):
  $D_x^2 = \frac{1}{N}\sum_{i=1}^{N} (x_i - m)^2$
* Выборочная дисперсия при **неизвестном** среднем, формула (77):
  $D_x^2 = \frac{1}{N}\sum_{i=1}^{N} (x_i - m_x)^2$

In [ ]:
def sample_mean(x):                       # (75)
    return np.sum(x) / len(x)


def sample_var_known_mean(x, m_true):     # (76)
    return np.sum((x - m_true) ** 2) / len(x)


def sample_var(x):                        # (77)
    mx = sample_mean(x)
    return np.sum((x - mx) ** 2) / len(x)


def moments_table(title, x, m_theor, var_theor):
    mx = sample_mean(x)
    d76 = sample_var_known_mean(x, m_theor)
    d77 = sample_var(x)
    def rel(est, th):
        return abs(est - th) / abs(th) * 100

    print(title)
    print("-" * 72)
    print(f"{'Характеристика':<34}{'Теория':>12}{'Выборка':>12}{'Погр., %':>12}")
    print(f"{'Среднее m_x (75)':<34}{m_theor:>12.4f}{mx:>12.4f}{rel(mx, m_theor):>12.2f}")
    print(f"{'Дисперсия D² (76), m известно':<34}{var_theor:>12.4f}{d76:>12.4f}{rel(d76, var_theor):>12.2f}")
    print(f"{'Дисперсия D² (77), m неизвестно':<34}{var_theor:>12.4f}{d77:>12.4f}{rel(d77, var_theor):>12.2f}")
    print()
    return mx, d77

## Гистограмма и полигон накопленных частот

Построение выполняется строго по пунктам 6 и 7 задания:

1. Выборка упорядочивается.
2. Число интервалов группировки $k = (4\ldots5)\lg N$ при $9 < k < 21$. Для симметричного
   распределения $k$ берётся нечётным.
3. Границы: $d = 1.02\,(x_{max} - x_{min})$, $\Delta x = d / k$, левая граница с запасом
   1% от размаха. Для одностороннего распределения (Рэлей) $a = 0$.
4. $\Delta_i = (a + (i-1)\Delta x,\ a + i\Delta x)$. Подсчитывается $k_i$ — число элементов в интервале.
5. Относительные частоты $\omega_i = k_i / (N\Delta x)$ дают высоты столбцов гистограммы.
   Сравниваются с теоретической $w(x)$ в центрах интервалов $a + (i - 0.5)\Delta x$.
6. Накопленные частоты $F_q = \sum_{i=1}^{q} k_i / N$ строятся ступенькой на $\Delta_q$.
   Сравниваются с теоретической $F(x)$ в концах интервалов.

In [ ]:
def grouping(x, symmetric, one_sided=False):
    x = np.sort(x)                                   # 6a
    N = len(x)

    k = round(5 * math.log10(N))                     # 6b
    k = min(max(k, 10), 20)
    if symmetric and k % 2 == 0:
        k -= 1

    x_min, x_max_s = x[0], x[-1]                      # 6c
    if one_sided:
        left = 0.0
        d = 1.02 * x_max_s
    else:
        d = 1.02 * (x_max_s - x_min)
        left = x_min - 0.01 * (x_max_s - x_min)
    dx = d / k

    edges = left + dx * np.arange(k + 1)             # 6d
    idx = np.minimum(((x - left) // dx).astype(int), k - 1)
    counts = np.bincount(idx, minlength=k)           # 6e

    omega = counts / (N * dx)                        # 6f
    F_q = np.cumsum(counts) / N                      # 7b, 7c
    return k, edges, dx, counts, omega, F_q


def plot_distribution(x, pdf, cdf, title, symmetric, one_sided=False):
    k, edges, dx, counts, omega, F_q = grouping(x, symmetric, one_sided)
    centers = edges[:-1] + dx / 2
    xs = np.linspace(edges[0], edges[-1], 600)

    print(f"{title}: k = {k}, Δx = {dx:.4f}, min k_i = {counts.min()}, max k_i = {counts.max()}")
    dev_pdf = np.max(np.abs(omega - pdf(centers)))
    dev_cdf = np.max(np.abs(F_q - cdf(edges[1:])))
    print(f"  max |ω_i − w(центр)| = {dev_pdf:.4f},  max |F_q − F(конец)| = {dev_cdf:.4f}")

    _, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.bar(centers, omega, width=dx, alpha=0.55, edgecolor="black",
            label="Гистограмма ω_i = k_i / (NΔx)")
    ax1.plot(xs, pdf(xs), "r", lw=2, label="Теоретическая w(x)")
    ax1.plot(centers, pdf(centers), "ro", ms=4, label="w(x) в центрах интервалов")
    ax1.set_title(f"{title}: гистограмма")
    ax1.set_xlabel("x")
    ax1.set_ylabel("w(x)")
    ax1.grid(alpha=0.4)
    ax1.legend()

    ax2.stairs(F_q, edges, lw=2, label="Полигон накопленных частот F_q")
    ax2.plot(xs, cdf(xs), "r", lw=2, label="Теоретическая F(x)")
    ax2.plot(edges[1:], cdf(edges[1:]), "ro", ms=4, label="F(x) в концах интервалов")
    ax2.set_title(f"{title}: полигон накопленных частот")
    ax2.set_xlabel("x")
    ax2.set_ylabel("F(x)")
    ax2.grid(alpha=0.4)
    ax2.legend(loc="lower right")

    plt.tight_layout()
    plt.show()

## 1. Равномерное распределение — метод обратной функции

Функция распределения и плотность на $(a, b)$, формула (65):

$$F(x) = \frac{x - a}{b - a}, \qquad w(x) = \frac{1}{b - a}.$$

Теоретические моменты: $m_x = \dfrac{a + b}{2}$ (66), $\sigma_x^2 = \dfrac{(b - a)^2}{12}$ (67).

Обратная функция находится явно, формула (68): $x_i = G(r_i) = r_i\,(b - a) + a$.

In [ ]:
def generate_uniform(a, b, N):
    r = rng.random(N)             # базовые числа r_i ∈ (0, 1)
    return r * (b - a) + a        # (68)


def uniform_pdf(x):
    return np.where((x >= a) & (x <= b), 1 / (b - a), 0.0)


def uniform_cdf(x):
    return np.clip((x - a) / (b - a), 0.0, 1.0)


x_uni = generate_uniform(a, b, N)
m_uni = (a + b) / 2               # (66)
var_uni = (b - a) ** 2 / 12       # (67)

res_uni = moments_table("Равномерное распределение", x_uni, m_uni, var_uni)
plot_distribution(x_uni, uniform_pdf, uniform_cdf, "Равномерный закон", symmetric=True)

## 2. Гауссовское распределение — на основе ЦПТ

Функция распределения (73) не выражается через элементарные функции, поэтому метод обратной
функции неприменим. По ЦПТ сумма $v = \sum_{i=1}^{n} r_i$ распределена приближённо нормально
с $m_v = n/2$ и $D_v^2 = n/12$. Нормируем:

$$\xi = \frac{v - m_v}{D_v} = \sqrt{\frac{12}{n}}\sum_{i=1}^{n}(r_i - 0.5) \;\sim\; N(0, 1),$$

и переходим к произвольным параметрам: $X = D\,\xi + m$.

В методичке в знаменателе формулы нормировки напечатано $D_v^2$. Делить нужно на СКО $D_v = \sqrt{n/12}$,
иначе дисперсия $\xi$ не будет равна 1. Итоговая формула $\xi = \sqrt{12/n}\sum(r_i - 0.5)$ в методичке верная.

Теоретические кривые — формулы (73), (74). Интеграл (73) вычисляется через функцию ошибок:
$F(x) = \tfrac12\left[1 + \operatorname{erf}\!\left(\tfrac{x - m}{D\sqrt2}\right)\right]$.
Коэффициент использования датчика равен $1/n$.

В крайних интервалах гистограммы (хвосты гауссианы) $k_i$ может оказаться меньше
рекомендуемых 5…10. Это естественно: вероятность попасть в хвосты мала.

In [ ]:
def generate_gaussian_clt(m, D, N, n):
    r = rng.random((N, n))                              # n базовых чисел на одно значение
    xi = math.sqrt(12 / n) * np.sum(r - 0.5, axis=1)    # стандартная N(0, 1)
    return D * xi + m


_erf = np.vectorize(math.erf)


def gauss_pdf(x):                                        # (74)
    return np.exp(-(x - m) ** 2 / (2 * D ** 2)) / math.sqrt(2 * math.pi * D ** 2)


def gauss_cdf(x):                                        # (73)
    return 0.5 * (1 + _erf((x - m) / (D * math.sqrt(2))))


x_gauss = generate_gaussian_clt(m, D, N, n_clt)

print(f"Использовано базовых чисел: {N * n_clt}, коэффициент использования датчика 1/n = {1 / n_clt:.3f}\n")
res_gauss = moments_table(f"Гауссовское распределение N({m}, {D}²), ЦПТ, n = {n_clt}", x_gauss, m, D ** 2)
plot_distribution(x_gauss, gauss_pdf, gauss_cdf, "Гауссовский закон (ЦПТ)", symmetric=True)

## 3. Усечённое рэлеевское распределение — метод Неймана

Рэлеевский закон, формула (69):

$$F(x) = 1 - e^{-x^2 / 2b^2}, \qquad w(x) = \frac{x}{b^2}\, e^{-x^2 / 2b^2}, \quad x \ge 0.$$

Неусечённые моменты: $m_x = b\sqrt{\pi/2}$ (70), $\sigma_x^2 = (2 - \pi/2)\,b^2$ (71).

**Усечение.** Область значений ограничивается интервалом $(0, x_{max})$. Плотность усечённого закона
нормируется так, чтобы интеграл по интервалу был равен 1:

$$w_{ус}(x) = \frac{w(x)}{F(x_{max})}, \qquad F_{ус}(x) = \frac{F(x)}{F(x_{max})}, \quad 0 < x < x_{max}.$$

**Алгоритм Неймана.** Кривая $g(x) = w(x)$ вписывается в прямоугольник $(0, x_{max}) \times (0, M)$,
где $M = \max w(x) = w(b) = e^{-1/2}/b$ (максимум плотности Рэлея в точке $x = b$).

1. Формируются $R_1 = a + (b - a)r_i = x_{max}\,r_i$ и $R_2 = M r_j$ — координаты случайной точки $Q$ в прямоугольнике.
2. Если $R_2 \le g(R_1)$, точка лежит под кривой и $x_i = R_1$ принимается в выборку. Иначе точка отбрасывается.

Доля принятых точек равна отношению площади под кривой к площади прямоугольника: $F(x_{max}) / (M\,x_{max})$.

In [ ]:
def rayleigh_pdf(x):                                     # (69)
    return np.where(x >= 0, x / b_ray ** 2 * np.exp(-x ** 2 / (2 * b_ray ** 2)), 0.0)


def rayleigh_cdf(x):                                     # (69)
    return np.where(x >= 0, 1 - np.exp(-x ** 2 / (2 * b_ray ** 2)), 0.0)


F_trunc = float(rayleigh_cdf(x_max))                     # вероятность попасть в (0, x_max)


def trunc_pdf(x):
    return np.where((x >= 0) & (x <= x_max), rayleigh_pdf(x) / F_trunc, 0.0)


def trunc_cdf(x):
    return np.clip(rayleigh_cdf(x) / F_trunc, 0.0, 1.0)


def generate_rayleigh_neyman(N):
    M = float(rayleigh_pdf(min(b_ray, x_max)))           # высота прямоугольника
    accepted = []
    total = 0
    count = 0
    while count < N:
        R1 = x_max * rng.random(N)                       # x-координата точки Q
        R2 = M * rng.random(N)                           # y-координата точки Q
        ok = R2 <= rayleigh_pdf(R1)                      # точка под кривой g(x)
        accepted.append(R1[ok])
        count += ok.sum()
        total += N
    x = np.concatenate(accepted)[:N]
    return x, M, total, count


x_ray, M_ray, tries, hits = generate_rayleigh_neyman(N)

# Теоретические моменты усечённого закона — численным интегрированием w_ус(x)
grid = np.linspace(0, x_max, 200001)
m_trunc = np.trapezoid(grid * trunc_pdf(grid), grid)
var_trunc = np.trapezoid((grid - m_trunc) ** 2 * trunc_pdf(grid), grid)

m_full = b_ray * math.sqrt(math.pi / 2)                  # (70)
var_full = (2 - math.pi / 2) * b_ray ** 2                # (71)

print(f"M = max w(x) = {M_ray:.4f},  F(x_max) = {F_trunc:.4f}")
print(f"Сгенерировано точек Q: {tries}, под кривой: {hits}, в выборку взято первых {N}")
print(f"Доля принятых: опыт {hits / tries:.3f}, теория F(x_max)/(M·x_max) = {F_trunc / (M_ray * x_max):.3f}")
print(f"Неусечённый закон: m_x = {m_full:.4f} (70), σ² = {var_full:.4f} (71)\n")

res_ray = moments_table(f"Усечённое рэлеевское распределение, b = {b_ray}, (0, {x_max})",
                        x_ray, m_trunc, var_trunc)
plot_distribution(x_ray, trunc_pdf, trunc_cdf, "Усечённый рэлеевский закон (Нейман)",
                  symmetric=False, one_sided=True)

## 4. Сравнение выборочных и теоретических характеристик

In [ ]:
rows = [
    ("Равномерный (обр. функция)", m_uni, res_uni[0], var_uni, res_uni[1]),
    ("Гауссовский (ЦПТ)", m, res_gauss[0], D ** 2, res_gauss[1]),
    ("Рэлеевский усеч. (Нейман)", m_trunc, res_ray[0], var_trunc, res_ray[1]),
]

print(f"N = {N}\n")
print(f"{'Распределение':<28}{'m теор':>9}{'m выб':>9}{'δm,%':>7}{'D² теор':>10}{'D² выб':>10}{'δD²,%':>7}")
print("-" * 80)
for name, mt, ms, vt, vs in rows:
    print(f"{name:<28}{mt:>9.4f}{ms:>9.4f}{abs(ms - mt) / abs(mt) * 100:>7.2f}"
          f"{vt:>10.4f}{vs:>10.4f}{abs(vs - vt) / vt * 100:>7.2f}")

## Вывод

1. **Метод обратной функции** для равномерного закона даёт выборку, гистограмма которой близка
   к постоянной плотности $1/(b-a)$. Полигон накопленных частот повторяет линейную $F(x)$.
   Метод самый экономичный: одно базовое число на одно значение.
2. **Метод на основе ЦПТ** позволяет получить гауссовскую величину, хотя обратная функция
   к (73) в явном виде не существует. Уже при $n = 12$ гистограмма хорошо совпадает с
   теоретической кривой (74), а выборочные $m_x$ и $D_x^2$ отличаются от заданных
   $m = 6$, $D^2 = 4$ на доли процента. Цена — коэффициент использования датчика $1/n$.
   Кроме того, значения ограничены диапазоном $m \pm D\sqrt{3n}$, то есть «хвосты» распределения обрезаны.
3. **Метод Неймана** универсален: для него нужна только плотность $w(x)$ и её максимум $M$.
   Для усечённого рэлеевского закона часть точек отбрасывается (доля принятых
   $F(x_{max})/(M x_{max})$), поэтому датчик используется менее эффективно. Выборочные
   характеристики совпадают с теоретическими для **усечённого** закона. Отличие от
   неусечённых (70), (71) объясняется тем, что отброшен хвост $x > x_{max}$.
4. Выборочная дисперсия по формуле (76) (среднее известно) и по формуле (77) (среднее оценено
   по выборке) практически совпадают при $N = 6000$. Погрешность оценок всех моментов не превышает
   нескольких процентов и уменьшается с ростом $N$.